## Environment Setup

In [ ]:
!pip install open3d

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# GOOGLE_DRIVE_PATH = "/content/drive/MyDrive/THESIS"
DATASET_PATH = "/content/drive/MyDrive/THESIS_other/mmw/rsu_1"
MYDATASET_LIDAR_PATH = "/content/drive/MyDrive/THESIS_other/mmw/MyDataset_rsu1/lidar"
MYDATASET_LIDAR_LABEL_PATH = "/content/drive/MyDrive/THESIS_other/mmw/MyDataset_rsu1/labels_lidar"
MYDATASET_RADAR_PATH = "/content/drive/MyDrive/THESIS_other/mmw/MyDataset_rsu1/radar"
MYDATASET_RADAR_LABEL_PATH = "/content/drive/MyDrive/THESIS_other/mmw/MyDataset_rsu1/labels_radar"

In [ ]:
import numpy as np
import open3d as o3d
import json
import os
from sklearn.neighbors import KDTree

## Load lidar and radar file

In [ ]:
index = "017220"

# get lidar points
lidar_pcd_path = os.path.join(MYDATASET_LIDAR_PATH, index + ".pcd")
lidar_pcd = o3d.io.read_point_cloud(lidar_pcd_path)
lidar_points = np.asarray(lidar_pcd.points)
# get lidar labels
lidar_label_path = os.path.join(MYDATASET_LIDAR_LABEL_PATH, index + ".npy")
lidar_labels = np.load(lidar_label_path)

In [ ]:
# get radar json file
radar_json_path = os.path.join(DATASET_PATH, index + ".json")
with open(radar_json_path, "r") as f:
    radar_points_json = json.load(f)

# turn to pcd file
radar_points = []
for p in radar_points_json:
    r = p["depth"]
    az = p["azimuth"]
    alt = p["altitude"]

    x = r * np.cos(alt) * np.cos(az)
    y = r * np.cos(alt) * np.sin(az)
    z = r * np.sin(alt)

    radar_points.append([x, y, z])

radar_points = np.array(radar_points)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(radar_points)

# save raw lidar pcd to dataset
radar_pcd_save_path = os.path.join(MYDATASET_RADAR_PATH, f"{index}.pcd")
o3d.io.write_point_cloud(radar_pcd_save_path, pcd)

True

In [ ]:
tree = KDTree(lidar_points)
dist, ind = tree.query(radar_points, k=5)  # ind.shape = (num_radar_points, 5)

In [ ]:
N_radar_points = radar_points.shape[0]
radar_labels = np.zeros(N_radar_points, dtype=int)
dist_threshold = 4

for i in range(N_radar_points):
  close_lidar_labels = lidar_labels[ind[i]].copy()
  close_lidar_labels = close_lidar_labels[dist[i]<dist_threshold]
  if len(close_lidar_labels) == 0:
    radar_labels[i] = -1
  else:
    close_lidar_labels += 1
    radar_labels[i] = np.bincount(close_lidar_labels).argmax() - 1

# save radar point cloud labels
labels_radar_save_path = os.path.join(MYDATASET_RADAR_LABEL_PATH, f"{index}_tr{dist_threshold}.npy")
np.save(labels_radar_save_path, radar_labels)